# GridNet-HD — 2D branch: UPerNet training + per-image cache export (Phase 1, resume-safe)

เป้าหมาย: เทรน UPerNet (ConvNeXt-T encoder) บน image masks ของ train zones → วัด 2D mIoU บน val crops →
export cache ต่อภาพ (class probabilities @1/4 res + 32-ch features @1/8 res, fp16, ~50 MB/ภาพ) ลง Drive `GridNet/cache2d/<zone>/`

Runtime: **L4** (พอ) หรือ A100 (เร็วกว่า ~2 เท่า) · T4 ใช้ได้แต่ต้อง `CROP 768, BATCH 2`

| GPU | CROP | BATCH | ~s/it (โดยประมาณ) |
|---|---|---|---|
| A100 40/80 GB | 1024 | 4 | 0.4–0.6 (ถ้า WORKERS 8) |
| L4 24 GB | 1024 | 2 | 0.8 |
| T4 16 GB | 768 | 2 | 1.2 |

ถ้าเจอ CUDA out of memory ใน cell 6 → ลด BATCH หรือ CROP แล้ว **เปลี่ยน RUN_NAME** · v1 (8 zones) ImageVote ได้ 53.5 บน val เพราะไม่เห็นพื้นที่ t1/t4 ตอนเทรน → v2 ใช้ครบ 21 zones

ภาพ+mask ถูกโหลดจาก Hugging Face ครั้งแรกแล้วเก็บสำเนาไว้บน Drive `GridNet/raw2d/<zone>/` (session ถัดไป copy จาก Drive)
**Resume**: Run all ซ้ำได้เมื่อ session หลุด — เทรนต่อจาก `last.pth` · เปลี่ยน encoder/crop = เปลี่ยน RUN_NAME

In [ ]:
#@title 1. GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "no GPU runtime"


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
RAW_DRIVE = f'{DRIVE}/raw2d'                     # images + masks copied here after the first download
RUN_NAME = 'upernet_convnext_t_v2'               #@param {type:"string"}
CKPT_DIR = f'{DRIVE}/ckpt/{RUN_NAME}'
CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN_NAME}'    # one 2D cache per 2D model (v1 is at cache2d/<zone>, legacy)
TRAIN_ZONES = 't1z6a,t1z6b,t2z5,t3z3,t3z6,t3z7,t5a1,t5a3,t5a4,t5a5,t5b2,t5b3,t5b4,t5b6,t5c1,t5c2,t5c3,t6z2,t6z3,t6z4,t6z6'   #@param {type:"string"}
VAL_ZONES = 't5b5,t1z8,t4z1'                     #@param {type:"string"}
IMAGES_PER_ZONE = 200   #@param {type:"integer"}
ENCODER = 'convnext_tiny'   #@param {type:"string"}
CROP = 1024             #@param {type:"integer"}
BATCH = 4               #@param {type:"integer"}
WORKERS = 8             #@param {type:"integer"}
ITERS = 12000           #@param {type:"integer"}
MAX_HOURS = 4.0         #@param {type:"number"}
LR = 0.001              #@param {type:"number"}
EVAL_EVERY = 1000       #@param {type:"integer"}
EXPORT_ZONES = 't1z6a,t1z6b,t2z5,t3z3,t3z6,t3z7,t5a1,t5a3,t5a4,t5a5,t5b2,t5b3,t5b4,t5b6,t5c1,t5c2,t5c3,t6z2,t6z3,t6z4,t6z6'  #@param {type:"string"}
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RAW_DRIVE, exist_ok=True); os.makedirs(CACHE2D_DRIVE, exist_ok=True)
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 20 and (CROP > 768 or BATCH > 2):
    print(f"WARNING: {vram_gb:.0f} GB GPU -> use CROP 768, BATCH 2")
elif vram_gb < 30 and BATCH > 2:
    print(f"WARNING: {vram_gb:.0f} GB GPU -> BATCH 2 is the safe value at CROP 1024")
print("checkpoints ->", CKPT_DIR)


In [ ]:
#@title 3. Environment (restore tarball from Drive or build; + timm)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    if os.path.exists(TAR):
        subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
    else:
        open('/content/colab_env.sh', 'w').write(r'''#!/usr/bin/env bash
# Build a self-contained Python 3.10 + torch 2.4.1/cu124 environment with micromamba.
# Works on Colab (whatever system Python/torch it ships), RunPod and any Linux box with an
# NVIDIA driver >= 12.4. All heavy deps come as prebuilt wheels (no compilation).
#
#   bash colab_env.sh /content/env          # ~8-12 min on Colab
#   /content/env/bin/python -c "import spconv.pytorch, flash_attn, torch_scatter; print('ok')"
set -euo pipefail
ENV=${1:-/content/env}
TORCH=2.4.1
TV=0.19.1
CU=cu124

if [ ! -x "$ENV/bin/python" ]; then
  if ! command -v micromamba >/dev/null 2>&1; then
    mkdir -p /opt/mm && cd /opt/mm
    curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
    export PATH=/opt/mm/bin:$PATH
  fi
  micromamba create -y -q -p "$ENV" -c conda-forge python=3.10 pip git ninja
fi
PIP="$ENV/bin/python -m pip"
$PIP install -q --upgrade pip wheel setuptools packaging psutil
$PIP install -q torch==$TORCH torchvision==$TV --index-url https://download.pytorch.org/whl/$CU
$PIP install -q torch-scatter -f https://data.pyg.org/whl/torch-2.4.0+$CU.html
$PIP install -q spconv-$CU
# flash-attn: setup.py downloads the matching prebuilt wheel (torch 2.4 / cu12 / cp310 exists)
MAX_JOBS=4 $PIP install -q "flash-attn==2.7.4.post1" --no-build-isolation
$PIP install -q "numpy<2.3" scipy addict timm einops plyfile termcolor colorhash tensorboard
"$ENV/bin/python" - << 'EOF'
import torch, spconv.pytorch, torch_scatter, flash_attn
print("env ok | torch", torch.__version__, "| cuda", torch.version.cuda, "| spconv", spconv.__version__,
      "| flash_attn", flash_attn.__version__, "| gpu", torch.cuda.is_available())
EOF
''')
        p = subprocess.Popen(['bash', '/content/colab_env.sh', ENV], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            if 'FutureWarning' not in line: print(line, end='')
        p.wait(); assert p.returncode == 0
        os.makedirs(f'{DRIVE}/env', exist_ok=True)
        subprocess.run(['tar', '-C', '/content', '-czf', TAR, 'env'], check=True); print("env saved to Drive")
PY = f'{ENV}/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', 'timm', 'huggingface_hub', 'pillow'], check=True)
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/code'); import gridnet.train.engine2d, timm; print('imports ok, timm', timm.__version__)"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])


In [ ]:
#@title 5. Images + masks for the train/val zones (Drive copy if present, else Hugging Face -> Drive)
import os, shutil, subprocess, time
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True)
def fetch(zones):
    t0 = time.time()
    for z in zones:
        dst = f'{DATA}/{z}'
        if os.path.isdir(f'{dst}/images') and os.path.isdir(f'{dst}/masks'):
            continue
        src = f'{RAW_DRIVE}/{z}'
        if os.path.isdir(f'{src}/images') and os.path.isdir(f'{src}/masks'):
            shutil.copytree(src, dst, dirs_exist_ok=True); print(f"{z}: copied from Drive")
        else:
            code = ("import os,sys; sys.path.insert(0,'/content/code'); from gridnet.train.data2d import fetch_zones; "
                    f"fetch_zones(['{z}'], '{DATA}')")
            env = dict(os.environ, HF_HUB_ENABLE_HF_TRANSFER='0')
            try:
                from google.colab import userdata; env['HF_TOKEN'] = userdata.get('HF_TOKEN')   # optional (public dataset)
            except Exception: pass
            subprocess.run([PY, '-c', code], check=True, env=env); print(f"{z}: downloaded from Hugging Face")
            shutil.copytree(dst, src, dirs_exist_ok=True); print(f"{z}: copy saved to Drive")
        n = len(os.listdir(f'{dst}/images')); print(f"   {n} images  ({(time.time() - t0) / 60:.1f} min)")
zones = [s.strip() for s in (TRAIN_ZONES + ',' + VAL_ZONES).split(',') if s.strip()]
fetch(zones)
print(subprocess.run(['du', '-sh', DATA], capture_output=True, text=True).stdout.strip())


In [ ]:
#@title 6. Train (resumes automatically from last.pth)
import subprocess, time, os
cmd = [PY, '/content/code/scripts/train_upernet.py', '--data', DATA, '--ckpt-dir', CKPT_DIR,
       '--train-zones', TRAIN_ZONES, '--val-zones', VAL_ZONES, '--images-per-zone', str(IMAGES_PER_ZONE),
       '--encoder', ENCODER, '--crop', str(CROP), '--batch', str(BATCH), '--iters', str(ITERS), '--lr', str(LR),
       '--eval-every', str(EVAL_EVERY), '--max-hours', str(MAX_HOURS), '--workers', str(WORKERS)]
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore')
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
try:
    for line in p.stdout:
        if 'Warning' not in line: print(line, end='')
    p.wait(); print(f"exit {p.returncode} after {(time.time() - t0) / 3600:.2f} h")
except KeyboardInterrupt:
    p.terminate(); p.wait()
    print(f"stopped by user after {(time.time() - t0) / 60:.1f} min; last.pth on Drive is kept -> re-run this cell to resume")


In [ ]:
#@title 7. Evaluate best checkpoint on the fixed val crops
import subprocess, json, os
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore')
cmd = [PY, '/content/code/scripts/train_upernet.py', '--data', DATA, '--ckpt-dir', CKPT_DIR, '--val-zones', VAL_ZONES,
       '--encoder', ENCODER, '--crop', str(CROP), '--batch', str(BATCH), '--eval-only']
r = subprocess.run(cmd, capture_output=True, text=True, env=env)
print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l)[-2000:], r.stderr[-800:])
if os.path.exists(f'{CKPT_DIR}/eval2d.json'):
    e = json.load(open(f'{CKPT_DIR}/eval2d.json')); print({k: e[k] for k in ('ckpt', 'it', 'mIoU', 'OA')}); print(e['per_class'])


In [ ]:
#@title 8. Export the per-image 2D cache for EXPORT_ZONES -> Drive cache2d/<RUN_NAME> (resumable; compact uint8/int8 ~24 MB/image; ~1.7 s/image on A100, ~3-4 s on L4)
import subprocess, time, os
zones = [s.strip() for s in EXPORT_ZONES.split(',') if s.strip()]
fetch(zones)                                   # images of the export zones (cell 5 must have run)
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore')
cmd = [PY, '/content/code/scripts/train_upernet.py', '--data', DATA, '--ckpt-dir', CKPT_DIR, '--encoder', ENCODER,
       '--crop', str(CROP), '--batch', str(BATCH), '--export-cache', CACHE2D_DRIVE, '--export-zones', ','.join(zones)]
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
try:
    for line in p.stdout:
        if 'Warning' not in line: print(line, end='')
    p.wait(); print(f"exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
except KeyboardInterrupt:
    p.terminate(); p.wait(); print("stopped; re-run to continue (existing files are skipped)")


## หมายเหตุ
- cache ต่อภาพ: `prob` (11, H/4, W/4) fp16 = softmax ที่ความละเอียด output จริงของ UPerNet · `feat` (32, H/8, W/8) fp16 = bottleneck feature เฉลี่ย 2×2
- val mIoU ในนี้วัดบน crop 1024² ที่สุ่มแบบตายตัวจาก val images (ครึ่งหนึ่งมีคลาสโครงสร้างอยู่กลาง) ไม่ใช่ทั้งภาพ — ใช้เลือก checkpoint เท่านั้น
- จบแล้ว **Disconnect and delete runtime**